In [9]:
# =============================================================
# Replication of Coan et al. (2021)
# "Computer-assisted classification of contrarian claims
# about climate change"
# Paper: https://www.nature.com/articles/s41598-021-01714-4
# Code: https://github.com/traviscoan/cards
# Data: https://drive.google.com/uc?export=download&id=14exmlYCT3-K2byYHFFrShAIYiemJQroi
# =============================================================

import subprocess
import sys

# Install spaCy for text preprocessing (used in authors' preprocess.py)
subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy",
                      "--quiet"])

# Download spaCy English language model required by preprocess.py
subprocess.check_call([sys.executable, "-m", "spacy", "download",
                      "en_core_web_lg", "--quiet"])

subprocess.check_call([sys.executable, "-m", "pip", "install",
                      "--upgrade", "typing_extensions", "--quiet"])

# Download NLTK stopwords required by preprocess.py
import nltk
nltk.download('stopwords', quiet=True)

print("All dependencies installed successfully")

All dependencies installed successfully


In [11]:
import os
import sys

os.chdir(os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), 'cards'))

from utils import read_csv

# Import authors' own preprocessing module
import preprocess as pp

# Import authors' own logistic classifier function
from fit.logistic import fit_logistic_classifier

# sklearn imports for SVM and unweighted logistic models
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_score, recall_score, f1_score

import pandas as pd

print("All imports successful")

All imports successful


In [3]:
# training.csv contains 23,436 labelled paragraphs from
# conservative think-tanks and contrarian blogs (1998-2020)
train_data = read_csv("data/training/training.csv", remove_header=True)
print(f"Training data loaded: {len(train_data)} rows")

# Preprocess the text using authors' exact pipeline
# (This step can take a few minutes due to the spaCy processing)
print("Preprocessing training data (this will take 5-10 minutes)...")
tokens = [[pp.tokenize(pp.denoise_text(row[0]), remove_stops=True),
           row[1]] for row in train_data]
print(f"Preprocessing complete: {len(tokens)} paragraphs processed")

Training data loaded: 23436 rows
Preprocessing training data (this will take 5-10 minutes)...


C:\Users/Jordh Sidhu/Desktop/cards_replication/cards\preprocess.py:27: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


Preprocessing complete: 23436 paragraphs processed


In [4]:
# Load validation set (noisy) - used during model development
val_data = read_csv("data/training/validation.csv", remove_header=True)
print(f"Validation data loaded: {len(val_data)} rows")

# Load test set (noise free) - used for final evaluation
test_data = read_csv("data/training/test.csv", remove_header=True)
print(f"Test data loaded: {len(test_data)} rows")

# Store original labels for evaluation
y_val_original = [row[1] for row in val_data]
y_test_original = [row[1] for row in test_data]

print("Preprocessing validation and test data (this will take a few minutes)...")
X_val = [pp.tokenize(pp.denoise_text(row[0]), remove_stops=True)
         for row in val_data]
X_test = [pp.tokenize(pp.denoise_text(row[0]), remove_stops=True)
          for row in test_data]
print("Preprocessing complete")

Validation data loaded: 2605 rows
Test data loaded: 2904 rows
Preprocessing validation and test data (this will take a few minutes)...


C:\Users/Jordh Sidhu/Desktop/cards_replication/cards\preprocess.py:27: MarkupResemblesLocatorWarning: The input looks more like a URL than markup. You may want to use an HTTP client like requests to get the document behind the URL, and feed that document to Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


Preprocessing complete


In [5]:
# Train the Logistic (weighted) classifier 
print("Training Logistic (weighted) classifier...")
model_lw = fit_logistic_classifier(tokens)
print("Logistic (weighted) trained successfully")

Training Logistic (weighted) classifier...


C:\Users\Jordh Sidhu\Documents\ANACONDA\Lib\site-packages\sklearn\utils\_param_validation.py:558: FutureWarning: Passing an int for a boolean parameter is deprecated in version 1.2 and won't be supported anymore in version 1.4.
  warnings.warn(


Logistic (weighted) trained successfully


In [6]:
# Extract text and labels from training data
text_train = [row[0] for row in train_data]
labels_train = [row[1] for row in train_data]

# Vectorize to match those in the logistic classifier
vectorizer = TfidfVectorizer(min_df=3, max_features=None,
                             strip_accents='unicode',
                             ngram_range=(1, 2), use_idf=1,
                             smooth_idf=1, sublinear_tf=1)
X_train_vec = vectorizer.fit_transform(text_train)

# Encode string labels to integers for sklearn models
le = LabelEncoder()
y_train_enc = le.fit_transform(labels_train)

# Train Logistic (unweighted)
# (Unlike the other ones the authors did not publish hyperparameters for this model)
print("Training Logistic (unweighted)...")
clf_lu = LogisticRegression(C=7.96, solver='lbfgs',
                            multi_class='ovr', max_iter=200)
clf_lu.fit(X_train_vec, y_train_enc)
print("Logistic (unweighted) trained")

# Train SVM (unweighted)
# Linear SVM with standard hyperparameters
print("Training SVM (unweighted)...")
clf_svm_uw = LinearSVC(C=1.0, max_iter=1000)
clf_svm_uw.fit(X_train_vec, y_train_enc)
print("SVM (unweighted) trained")

# Train SVM (weighted)
# Linear SVM with class weighting to handle class imbalance
# (Same thing occurs with not publishing hyperparameters with this one and the unweighted one above)
print("Training SVM (weighted)...")
clf_svm_w = LinearSVC(C=1.0, max_iter=1000, class_weight='balanced')
clf_svm_w.fit(X_train_vec, y_train_enc)
print("All models trained successfully")

C:\Users\Jordh Sidhu\Documents\ANACONDA\Lib\site-packages\sklearn\utils\_param_validation.py:558: FutureWarning: Passing an int for a boolean parameter is deprecated in version 1.2 and won't be supported anymore in version 1.4.
  warnings.warn(


Training Logistic (unweighted)...
Logistic (unweighted) trained
Training SVM (unweighted)...
SVM (unweighted) trained
Training SVM (weighted)...
All models trained successfully


In [7]:
# Vectorize the validation and test sets
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

# Vectorize for logistic weighted model 
X_val_lw = model_lw['vectorizer'].transform(X_val)
X_test_lw = model_lw['vectorizer'].transform(X_test)

def get_metrics(y_true, y_pred_encoded, label_encoder):
    """
    Convert encoded predictions back to claim codes
    and calculate macro-averaged precision, recall and F1
    matching the reporting style of Table 2
    """
    y_pred = label_encoder.inverse_transform(y_pred_encoded)
    p = round(precision_score(y_true, y_pred, average="macro",
                              zero_division=0), 2)
    r = round(recall_score(y_true, y_pred, average="macro",
                           zero_division=0), 2)
    f = round(f1_score(y_true, y_pred, average="macro",
                       zero_division=0), 2)
    return p, r, f

# Evaluate the different methods- beginning with Logistic (weighted)
lw_val = get_metrics(y_val_original,
                     model_lw['clf'].predict(X_val_lw),
                     model_lw['label_encoder'])
lw_test = get_metrics(y_test_original,
                      model_lw['clf'].predict(X_test_lw),
                      model_lw['label_encoder'])

# Logistic (unweighted)
lu_val = get_metrics(y_val_original, clf_lu.predict(X_val_vec), le)
lu_test = get_metrics(y_test_original, clf_lu.predict(X_test_vec), le)

# SVM (unweighted)
svm_uw_val = get_metrics(y_val_original,
                         clf_svm_uw.predict(X_val_vec), le)
svm_uw_test = get_metrics(y_test_original,
                          clf_svm_uw.predict(X_test_vec), le)

# SVM (weighted)
svm_w_val = get_metrics(y_val_original,
                        clf_svm_w.predict(X_val_vec), le)
svm_w_test = get_metrics(y_test_original,
                         clf_svm_w.predict(X_test_vec), le)

print("All models evaluated successfully")

All models evaluated successfully


In [8]:
# Build results table to match the format of table 2 from the paper 

results = pd.DataFrame({
    'Model': [
        'Logistic (unweighted)',
        'Logistic (weighted)',
        'SVM (unweighted)',
        'SVM (weighted)'
    ],
    ('Validation set (noisy)', 'Precision'): [
        lu_val[0], lw_val[0], svm_uw_val[0], svm_w_val[0]],
    ('Validation set (noisy)', 'Recall'): [
        lu_val[1], lw_val[1], svm_uw_val[1], svm_w_val[1]],
    ('Validation set (noisy)', 'F1'): [
        lu_val[2], lw_val[2], svm_uw_val[2], svm_w_val[2]],
    ('Test set (noise free)', 'Precision'): [
        lu_test[0], lw_test[0], svm_uw_test[0], svm_w_test[0]],
    ('Test set (noise free)', 'Recall'): [
        lu_test[1], lw_test[1], svm_uw_test[1], svm_w_test[1]],
    ('Test set (noise free)', 'F1'): [
        lu_test[2], lw_test[2], svm_uw_test[2], svm_w_test[2]],
})

# Set model name as index
results = results.set_index('Model')

# Create multi-level column headers matching paper format
results.columns = pd.MultiIndex.from_tuples(results.columns)

# Display table
print("\nTable 2 Replication: Out-of-sample classification performance")
print("Replicating shallow classifier rows from Coan et al. (2021)\n")
print(results.to_string())
print("\nNote: Logistic (weighted) uses authors' exact published code.")
print("Other models use same TF-IDF features with standard hyperparameters")
print("as authors did not publish hyperparameters for these models.")


Table 2 Replication: Out-of-sample classification performance
Replicating shallow classifier rows from Coan et al. (2021)

                      Validation set (noisy)              Test set (noise free)             
                                   Precision Recall    F1             Precision Recall    F1
Model                                                                                       
Logistic (unweighted)                   0.72   0.45  0.52                  0.81   0.46  0.54
Logistic (weighted)                     0.62   0.69  0.65                  0.75   0.70  0.71
SVM (unweighted)                        0.71   0.49  0.55                  0.81   0.50  0.57
SVM (weighted)                          0.63   0.64  0.62                  0.75   0.65  0.68

Note: Logistic (weighted) uses authors' exact published code.
Other models use same TF-IDF features with standard hyperparameters
as authors did not publish hyperparameters for these models.
